## Тема: Эмуляция distributed-окружения, замер коллективных примитивов и отладка PyTorch DDP

---

### 📍 Введение и цели лабораторной работы


# Практическая работа №1: Основы распределённых вычислений в PyTorch
Курс: Распределённые вычисления: обучение и инференс  
Среда выполнения: Google Colab (Free / Pro)  
Бэкенды: `Gloo` (CPU / Multi-process), `NCCL` (GPU)

### 🎯 Цели практики:
1. Научиться программно профилировать расход VRAM и рассчитывать память под веса, градиенты и оптимизатор Adam.
2. Эмулировать многопроцессный распределённый кластер в Colab с помощью `torchrun` и `%%writefile`.
3. Измерить пропускную способность коллективных примитивов (`AllReduce`, `AllGather`, `ReduceScatter`).
4. Реализовать PyTorch DDP с корректным отключением синхронизаций через `model.no_sync()`.
5. Смоделировать и устранить сетевое зависание (Deadlock / Timeout) через настройки `NCCL_DEBUG`.

In [1]:
# ==========================================
# 1. Проверка окружения и доступных GPU
# ==========================================
import os
import sys
import torch

print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {gpu_name} | VRAM: {vram_gb:.2f} GB")
    DEVICE_TYPE = "cuda"
    BACKEND = "nccl"
else:
    print("GPU не обнаружена. Запуск в режиме эмуляции на CPU (Gloo backend).")
    DEVICE_TYPE = "cpu"
    BACKEND = "gloo"

print(f"Целевой бэкенд для distributed: '{BACKEND}'")

Python version: 3.12.13
PyTorch version: 2.11.0+cpu
CUDA Available: False
GPU не обнаружена. Запуск в режиме эмуляции на CPU (Gloo backend).
Целевой бэкенд для distributed: 'gloo'


### 📍 Упражнение 1. Калькулятор VRAM и профилирование памяти
---
## 📐 Упражнение 1: Программный калькулятор VRAM и профилирование
Рассчитаем статическое потребление памяти моделью Transformer при обучении в точностях FP32, FP16/BF16 и подтердим расчёт нативным профилировщиком PyTorch.

In [2]:
# ==========================================
# 2. Программный калькулятор VRAM
# ==========================================
def calculate_model_memory(num_params_billions: float, precision: str = "bf16"):
    """
    Рассчитывает потребление VRAM (в ГБ) для обучения и инференса.
    """
    phi = num_params_billions
    bytes_per_param = 2 if precision in ["fp16", "bf16"] else 4

    # Статическая память при обучении (Adam FP32)
    weights_mem = phi * bytes_per_param
    grads_mem = phi * bytes_per_param
    adam_states_mem = phi * 12  # Master weights (4B) + Momentum (4B) + Variance (4B)

    total_train_mem = weights_mem + grads_mem + adam_states_mem
    total_inf_mem = weights_mem

    print(f"=== Расчет памяти для модели {phi}B параметров ({precision.upper()}) ===")
    print(f"1. Веса модели:       {weights_mem:.2f} GB")
    print(f"2. Градиенты:         {grads_mem:.2f} GB")
    print(f"3. Состояния Adam:    {adam_states_mem:.2f} GB")
    print(f"----------------------------------------")
    print(f"ИТОГО под параметры при обучении: {total_train_mem:.2f} GB")
    print(f"ИТОГО под параметры при инференсе: {total_inf_mem:.2f} GB\n")

# Пример расчета для моделей 1B, 7B и 70B
calculate_model_memory(1.0, "bf16")
calculate_model_memory(7.0, "bf16")

# Практическое замерение выделенной памяти в PyTorch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    # Создаем тестовую модель (Linear layer ~ 100M params)
    linear = torch.nn.Linear(8192, 12288, bias=False).to("cuda").to(torch.bfloat16)

    allocated_mb = torch.cuda.memory_allocated() / (1024**2)
    print(f"[PyTorch Check] Выделено под веса Linear (100M): {allocated_mb:.2f} MB")
    del linear
    torch.cuda.empty_cache()

=== Расчет памяти для модели 1.0B параметров (BF16) ===
1. Веса модели:       2.00 GB
2. Градиенты:         2.00 GB
3. Состояния Adam:    12.00 GB
----------------------------------------
ИТОГО под параметры при обучении: 16.00 GB
ИТОГО под параметры при инференсе: 2.00 GB

=== Расчет памяти для модели 7.0B параметров (BF16) ===
1. Веса модели:       14.00 GB
2. Градиенты:         14.00 GB
3. Состояния Adam:    84.00 GB
----------------------------------------
ИТОГО под параметры при обучении: 112.00 GB
ИТОГО под параметры при инференсе: 14.00 GB



## 🚀 Упражнение 2: Замер скорости коллективных примитивов
Мы создадим отдельный Python-скрипт `bench_primitives.py` и запустим его через `torchrun` с 2 параллельными процессами (ranks), чтобы сравнить время выполнения операций `AllReduce`, `AllGather` и `ReduceScatter`.

### 📍 Генерация скрипта `bench_primitives.py`

In [3]:
%%writefile bench_primitives.py
import os
import time
import torch
import torch.distributed as dist

def main():
    # Инициализация распределённой группы
    use_cuda = torch.cuda.is_available()
    backend = "nccl" if use_cuda else "gloo"
    dist.init_process_group(backend=backend)

    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    world_size = int(os.environ.get("WORLD_SIZE", 1))

    device = torch.device(f"cuda:{local_rank}" if use_cuda else "cpu")
    if use_cuda:
        torch.cuda.set_device(device)

    # Тензор размером 50 MB (12.5 млн float32 чисел)
    tensor_size = 12_500_000
    tensor = torch.ones(tensor_size, device=device) * (local_rank + 1)

    # 1. Замер AllReduce
    dist.barrier()
    start_time = time.time()
    dist.all_reduce(tensor, op=dist.ReduceOp.SUM)
    if use_cuda: torch.cuda.synchronize()
    allreduce_time = time.time() - start_time

    # 2. Замер AllGather
    gather_list = [torch.zeros(tensor_size, device=device) for _ in range(world_size)]
    dist.barrier()
    start_time = time.time()
    dist.all_gather(gather_list, tensor)
    if use_cuda: torch.cuda.synchronize()
    allgather_time = time.time() - start_time

    if local_rank == 0:
        data_size_mb = (tensor_size * 4) / (1024**2)
        print(f"\n=== Результаты замера бенчмарка (World Size: {world_size}) ===")
        print(f"Размер пересылаемого тензора: {data_size_mb:.2f} MB")
        print(f"1. AllReduce Time:  {allreduce_time*1000:.2f} ms | Val Check: {tensor[0].item()}")
        print(f"2. AllGather Time:  {allgather_time*1000:.2f} ms | Gathered Ranks: {len(gather_list)}")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()

Writing bench_primitives.py


In [4]:
# ==========================================
# 3. Запуск бенчмарка на 2 процессах
# ==========================================
!torchrun --nproc_per_node=2 bench_primitives.py

W0808 07:45:03.584000 2066 torch/distributed/run.py:851] 
W0808 07:45:03.584000 2066 torch/distributed/run.py:851] *****************************************
W0808 07:45:03.584000 2066 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0808 07:45:03.584000 2066 torch/distributed/run.py:851] *****************************************

=== Результаты замера бенчмарка (World Size: 2) ===
Размер пересылаемого тензора: 47.68 MB
1. AllReduce Time:  43.91 ms | Val Check: 3.0
2. AllGather Time:  193.40 ms | Gathered Ranks: 2


## ⚡ Упражнение 3: Проверка эффективности `model.no_sync()`
При аккумуляции градиентов синхронизация `AllReduce` нужна **только на финальном микро-шаге**. Сравним время выполнения 4 шагов аккумулирования **без** `no_sync()` и **с использованием** `no_sync()`.

In [5]:
%%writefile bench_no_sync.py
import os
import time
import torch
import torch.distributed as dist
import torch.nn as nn
from torch.nn.parallel import DistributedDataParallel as DDP

def main():
    use_cuda = torch.cuda.is_available()
    backend = "nccl" if use_cuda else "gloo"
    dist.init_process_group(backend=backend)

    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    device = torch.device(f"cuda:{local_rank}" if use_cuda else "cpu")
    if use_cuda: torch.cuda.set_device(device)

    # Создаем тяжелую полносвязную модель
    model = nn.Sequential(
        nn.Linear(4096, 4096),
        nn.ReLU(),
        nn.Linear(4096, 4096)
    ).to(device)

    ddp_model = DDP(model, device_ids=[local_rank] if use_cuda else None)
    optimizer = torch.optim.SGD(ddp_model.parameters(), lr=0.01)

    accum_steps = 4
    inputs = [torch.randn(32, 4096, device=device) for _ in range(accum_steps)]

    # ---- ТЕСТ A: БЕЗ no_sync() (AllReduce на каждом шаге) ----
    dist.barrier()
    start_time = time.time()
    optimizer.zero_grad()
    for x in inputs:
        outputs = ddp_model(x)
        loss = outputs.sum()
        loss.backward()  # Вызывает AllReduce 4 раза!
    optimizer.step()
    if use_cuda: torch.cuda.synchronize()
    time_without_nosync = time.time() - start_time

    # ---- ТЕСТ B: С no_sync() (AllReduce ТОЛЬКО на 4-м шаге) ----
    dist.barrier()
    start_time = time.time()
    optimizer.zero_grad()
    for i in range(accum_steps - 1):
        with ddp_model.no_sync():  # Отключаем сеть для первых 3 шагов
            outputs = ddp_model(inputs[i])
            loss = outputs.sum()
            loss.backward()

    # Финальный микро-шаг запускает AllReduce
    outputs = ddp_model(inputs[-1])
    loss = outputs.sum()
    loss.backward()
    optimizer.step()
    if use_cuda: torch.cuda.synchronize()
    time_with_nosync = time.time() - start_time

    if local_rank == 0:
        print(f"\n=== Сравнение производительности Gradient Accumulation ({accum_steps} шагов) ===")
        print(f"1. Время БЕЗ no_sync(): {time_without_nosync*1000:.2f} ms")
        print(f"2. Время С no_sync():  {time_with_nosync*1000:.2f} ms")
        speedup = ((time_without_nosync - time_with_nosync) / time_without_nosync) * 100
        print(f"🚀 Ускорение от отмены лишних синхронизаций: {speedup:.1f}%")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()

Writing bench_no_sync.py


In [6]:
# ==========================================
# 4. Запуск бенчмарка аккумуляции градиентов
# ==========================================
!torchrun --nproc_per_node=2 bench_no_sync.py

W0808 07:50:25.769000 3365 torch/distributed/run.py:851] 
W0808 07:50:25.769000 3365 torch/distributed/run.py:851] *****************************************
W0808 07:50:25.769000 3365 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0808 07:50:25.769000 3365 torch/distributed/run.py:851] *****************************************

=== Сравнение производительности Gradient Accumulation (4 шагов) ===
1. Время БЕЗ no_sync(): 3063.33 ms
2. Время С no_sync():  2018.31 ms
🚀 Ускорение от отмены лишних синхронизаций: 34.1%


## 🛠️ Упражнение 4: Симуляция Deadlock и обработка таймаутов
Симулируем ситуацию, когда **Rank 1 зависает или выполняет слишком долгие вычисления**, а Rank 0 ждет его в `AllReduce`. Мы настроим явный таймаут и логирование `NCCL_DEBUG`, чтобы штатно перехватить ошибку без вечного зависания скрипта.

In [7]:
%%writefile timeout_demo.py
import os
import time
import datetime
import torch
import torch.distributed as dist

def main():
    use_cuda = torch.cuda.is_available()
    backend = "nccl" if use_cuda else "gloo"

    # Устанавливаем короткий таймаут 3 секунды для предотвращения вечного повисания
    try:
        dist.init_process_group(
            backend=backend,
            timeout=datetime.timedelta(seconds=3)
        )
    except Exception as e:
        print(f"Ошибка инициализации: {e}")
        return

    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    device = torch.device(f"cuda:{local_rank}" if use_cuda else "cpu")

    tensor = torch.ones(100, device=device)

    if local_rank == 1:
        print(f"[Rank 1] Симулируем задержку вычислений (зависание) на 5 секунд...")
        time.sleep(5)  # Превышает таймаут 3 секунды!

    print(f"[Rank {local_rank}] Вход в точку синхронизации barrier()...")

    try:
        dist.barrier()
        print(f"[Rank {local_rank}] Успешно прошел barrier!")
    except Exception as err:
        print(f"\n❌ [Rank {local_rank}] ПЕРЕХВАТЕН ТАЙМАУТ СЕТИ!")
        print(f"Детали ошибки: {type(err).__name__}")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()

Writing timeout_demo.py


In [8]:
# ==========================================
# 5. Запуск отладочного теста с логированием
# ==========================================
os.environ["NCCL_DEBUG"] = "WARN"
os.environ["NCCL_IB_DISABLE"] = "1"

!torchrun --nproc_per_node=2 timeout_demo.py

W0808 07:57:38.974000 5100 torch/distributed/run.py:851] 
W0808 07:57:38.974000 5100 torch/distributed/run.py:851] *****************************************
W0808 07:57:38.974000 5100 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0808 07:57:38.974000 5100 torch/distributed/run.py:851] *****************************************
[Rank 0] Вход в точку синхронизации barrier()...[Rank 1] Симулируем задержку вычислений (зависание) на 5 секунд...


❌ [Rank 0] ПЕРЕХВАТЕН ТАЙМАУТ СЕТИ!
Детали ошибки: RuntimeError
[Rank 1] Вход в точку синхронизации barrier()...

❌ [Rank 1] ПЕРЕХВАТЕН ТАЙМАУТ СЕТИ!
Детали ошибки: RuntimeError


## 🎯 Итоговый чек-лист выполнения Практики №1:

- [x] **Профилирование памяти:** Освоен расчёт VRAM для параметров, градиентов и Adam ($16\Phi$ B).
- [x] **Запуск процессов:** Успешно эмулирована работа multi-rank кластера через `torchrun`.
- [x] **Коллективные примитивы:** Проведены замеры задержек `AllReduce` и `AllGather`.
- [x] **Оптимизация DDP:** Подтверждено ускорение отмена лишних вызовов `AllReduce` через `model.no_sync()`.
- [x] **Траблшутинг:** Настроена защита от вечного повисания процессов через `timeout` и `NCCL_DEBUG`.

---
### 💡 Задание для самостоятельной работы:
Измените скрипт `bench_no_sync.py`, добавив проверку применения скалера градиентов `torch.cuda.amp.GradScaler()` при использовании точности `FP16`. Убедитесь, что `scaler.step(optimizer)` корректно вызывается только после объединения всех накопленных градиентов.

### 🛠️ Инструкция по использованию:
1. Откройте [Google Colab](https://colab.research.google.com/).
2. Создайте новый ноутбук.
3. Последовательно скопируйте ячейки выше и нажмите **Runtime → Run all** (`Ctrl + F9`).
4. Ноутбук автоматически определит наличие T4 GPU или переключится на CPU-эмуляцию с бэкендом `Gloo`, выполнив все тесты за время < 1 минуты.